# Lab 32: The Word Frequency Shortcut

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 4, *The Language Engine*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-32.ipynb)

**What you will do:** time yourself reading a list of common words against a list of rare words, compare your speed-up with the typical 20-30% the book reports, feed the bigram model different training text to see how a purely frequency-based "language model" changes its predictions, and use a real subword tokeniser to see the same frequency effect at work inside a large language model.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 32 you timed yourself reading aloud a list of common words ("table," "water," "house")
and a list of rare words ("aperture," "flange," "epaulet"). The common-word list is typically
read 20-30% faster, even though both lists contain real English words of similar length. Your
brain recognises frequent words as whole units, the way it recognises a familiar face, while
rare words must be worked out piece by piece.

## Record your results

Time yourself reading each list aloud, clearly and without errors, and count any misreadings.
Lab 32 uses 20 words per list; adjust `items_per_list` if you used a different number.

In [ ]:
items_per_list = 20
my_times_s = {"A: common words": 8.4, "B: rare words": 12.9}       # example data: replace with yours
my_errors = {"A: common words": 0, "B: rare words": 2}             # example data

results = pd.DataFrame({"time (s)": my_times_s, "errors": my_errors})
results["s per word"] = (results["time (s)"] / items_per_list).round(2)
display(results)
results["s per word"].plot.barh(color=["steelblue", "firebrick"], figsize=(6, 2.2))
plt.xlabel("seconds per word"); plt.gca().invert_yaxis(); plt.show()

## Compare

Lab 32 reports that the common-word list is typically read 20-30% faster than the rare-word
list. Compute your own percentage speed-up and see where it falls.

In [ ]:
common_s, rare_s = results.loc["A: common words", "s per word"], results.loc["B: rare words", "s per word"]
speedup = (rare_s - common_s) / rare_s
print(f"Your speed-up on common words: {speedup:.0%}")
if 0.20 <= speedup <= 0.30:
    print("Inside the typical 20-30% range Lab 32 reports.")
elif speedup > 0.30:
    print("Above the typical range -- perhaps the rare-word list included some genuinely"
          " unfamiliar words for you, exaggerating the gap.")
else:
    print("Below the typical range -- perhaps you already knew the 'rare' words well, or"
          " read the common-word list unusually slowly.")

## Simulation

Your brain reads words faster when they are common. This simulation builds the simplest possible "language model"—one that predicts the next character based on letter-pair frequencies in English—and shows how even this crude model captures real patterns in the language.

In [ ]:
import numpy as np
from collections import Counter

# A tiny corpus (try replacing with your own text)
corpus = ("the brain is a prediction machine that generates models of the world "
          "and updates them only when surprised the prediction error is the signal that drives "
          "all learning in both biological brains and artificial neural networks")

# Build bigram (letter-pair) frequency table
bigrams = Counter(corpus[i:i+2] for i in range(len(corpus) - 1))
chars = sorted(set(corpus))

# Predict next character given current character
def predict_next(current_char, top_n=5):
    candidates = {c: bigrams.get(current_char + c, 0) for c in chars}
    total = sum(candidates.values()) or 1
    ranked = sorted(candidates.items(), key=lambda x: -x[1])[:top_n]
    return [(c, count / total) for c, count in ranked]

# Show predictions for a few seed characters
seeds = ["t", "p", "e", " "]
for seed in seeds:
    preds = predict_next(seed)
    top = ", ".join(f"'{c}' ({p:.0%})" for c, p in preds)
    print(f"After '{seed}' → {top}")

# Generate text by sampling from the model
np.random.seed(0)
generated = "the "
for _ in range(60):
    preds = predict_next(generated[-1])
    chars_list, probs = zip(*preds)
    probs = np.array(probs)
    probs /= probs.sum()
    generated += np.random.choice(chars_list, p=probs)

print(f"\nGenerated text: {generated}")

**What to notice:** After "t", the model predicts "h" with about 50% probability—because "th" is the most common English bigram in this sample. The generated text is gibberish, but it contains recognisable English-like fragments. Real language models (GPT, Claude) use the same logic—next-element prediction—but over billions of parameters instead of 26 letters, and over words and sub-word tokens rather than single characters.

## Change the parameters

Edit **new_corpus** to change the training text (try a single short sentence, a much
longer passage, or text in a different style), pick a **seed** letter to predict from,
and adjust **gen_length** and **top_n**. The function below rebuilds the bigram table
from your text using the same `Counter`-based approach as the cell above.
(If the widgets do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, Textarea, Dropdown, IntSlider

def explore(new_corpus=corpus, seed="t", gen_length=80, top_n=5):
    text = new_corpus.lower()
    bigrams_local = Counter(text[i:i + 2] for i in range(len(text) - 1))
    chars_local = sorted(set(text))

    def predict_next_local(current_char, n=top_n):
        candidates = {c: bigrams_local.get(current_char + c, 0) for c in chars_local}
        total = sum(candidates.values()) or 1
        ranked = sorted(candidates.items(), key=lambda x: -x[1])[:n]
        return [(c, count / total) for c, count in ranked]

    if seed not in chars_local:
        print(f"The character '{seed}' does not appear in this text -- try one that does.")
        return

    preds = predict_next_local(seed)
    top = ", ".join(f"'{c}' ({p:.0%})" for c, p in preds)
    print(f"After '{seed}' -> {top}")

    np.random.seed(0)
    generated = seed
    for _ in range(gen_length):
        preds = predict_next_local(generated[-1])
        chars_list, probs = zip(*preds)
        probs = np.array(probs)
        probs /= probs.sum()
        generated += np.random.choice(chars_list, p=probs)
    print(f"\nGenerated text: {generated}")

interact(explore,
         new_corpus=Textarea(value=corpus, description="Corpus:", layout={"width": "600px", "height": "80px"}),
         seed=Dropdown(options=["t", "p", "e", " ", "a", "i"], value="t"),
         gen_length=IntSlider(value=80, min=20, max=300, step=20),
         top_n=IntSlider(value=5, min=1, max=10, step=1));

## The AI side

Large language models do not read letter by letter or word by word; they read **tokens**,
units produced by a subword tokeniser such as Byte-Pair Encoding, or BPE (Sennrich, Haddow,
& Birch, 2016). Common words are usually a single token; rare words are split into several.
Below, GPT-2's tokeniser (just the tokeniser, not the full model -- no heavy download) counts
tokens for your two word lists, so you can see the same frequency effect that slowed your own
reading show up as a difference in how many pieces a word is cut into.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
common_words = ["table", "water", "house", "doctor", "music", "garden", "school", "money", "travel", "family"]
rare_words = ["aperture", "flange", "caulk", "epaulet", "reticule", "vestibule", "plinth", "grommet", "sconce", "lintel"]

def token_counts(words):
    return pd.DataFrame({
        "word": words,
        "tokens": [tokenizer.tokenize(" " + w) for w in words],
        "n_tokens": [len(tokenizer.tokenize(" " + w)) for w in words],
    })

common_df, rare_df = token_counts(common_words), token_counts(rare_words)
display(pd.concat([common_df.assign(list="common"), rare_df.assign(list="rare")], ignore_index=True))
print(f"Mean tokens per word, common list: {common_df.n_tokens.mean():.1f}")
print(f"Mean tokens per word, rare list:   {rare_df.n_tokens.mean():.1f}")

If the rare list needs more tokens per word on average, GPT-2 is showing a machine version
of your own reading-time effect: common words are handled as one efficient unit, rare words
cost more processing steps. The mechanisms are different in an important way: your slowdown
comes from decades of variable exposure to English text and speech, continuously updated
every time you read; GPT-2's tokeniser vocabulary was fixed once, by a statistical procedure
run over a training corpus, and never adapts to new text afterwards.

## Pool the class data

Pool the class's percentage speed-up (common list versus rare list), one row per person
(anonymous ID, never a name), and check whether the group mean falls inside the 20-30% range
Lab 32 reports.

In [ ]:
import io
csv_text = """id,common_s_per_word,rare_s_per_word
P01,0.42,0.60
P02,0.38,0.55
P03,0.50,0.64
P04,0.35,0.52
P05,0.44,0.58
"""  # example data: replace with your class CSV
df = pd.read_csv(io.StringIO(csv_text))
df["speedup"] = (df.rare_s_per_word - df.common_s_per_word) / df.rare_s_per_word

boot = [df.speedup.sample(len(df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean speed-up on common words: {df.speedup.mean():.0%} "
      f"(95% bootstrap interval {lo:.0%}-{hi:.0%}, n = {len(df)})")
df.speedup.plot.hist(bins=8, color="steelblue", alpha=0.7, figsize=(6, 3))
plt.axvspan(0.20, 0.30, color="grey", alpha=0.2, label="typical range (Lab 32)")
plt.xlabel("speed-up on common words"); plt.legend(); plt.show()

## Questions to think about

1. Raise top_n to 10. Does the top-ranked prediction after "t" change, or just how many alternatives are shown? What does this tell you about how concentrated English bigram probabilities are around a small number of likely letters?
2. Replace the corpus with a single short sentence copied several times, then with a single sentence copied only once, and rerun. Does the generated text still look English-like, or does it degrade into loops? Why would more training text usually help a frequency-based model like this one?
3. Set seed to a space character. What is the most likely next letter, and how does this connect to the lab's point that a handful of very high-frequency words (such as "the") dominate ordinary reading?
4. Paste in a corpus that repeats one long, unusual word many times (invent one if you like), and try seeding on one of its letters. Does the model start predicting that word's internal letter sequence confidently, even though the word is rare in English generally? What does that suggest about the difference between frequency within a text and frequency in the language as a whole?

## Challenge

Paste in a chunk of your own writing (an email, an essay, a text message) as the corpus and
compare the generated text and top predictions with the book's default corpus. This is the
same principle -- frequent exposure creates a processing shortcut -- that gives a fluent
reader of a particular style or language an advantage over an unfamiliar one.

In [ ]:
def next_char_entropy(text, seed_char):
    """Shannon entropy (in bits) of the letter that follows seed_char in this text.
    Lower entropy means the next letter is more predictable in this text."""
    text = text.lower()
    bigrams_local = Counter(text[i:i + 2] for i in range(len(text) - 1))
    chars_local = sorted(set(text))
    counts = np.array([bigrams_local.get(seed_char + c, 0) for c in chars_local])
    counts = counts[counts > 0]
    if counts.sum() == 0:
        return None
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs))

# Example data: replace with your own two text samples
my_text = "the quick brown fox jumps over the lazy dog again and again"  # example data
other_text = "xenon quartz vortex jazz onyx and lynx"                    # example data

for label, text in [("my_text", my_text), ("other_text", other_text)]:
    h = next_char_entropy(text, "t") if "t" in text else None
    print(f"{label}: entropy after 't' = {h}")
print("Lower entropy means the next letter is more predictable in that text -- a rough proxy")
print("for how strongly a pattern has been reinforced by exposure. Replace the example texts above.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-32.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")